# 💬 FLAN-T5 Dialogue Summarizer + RAG

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shalev396/ml-lab/blob/main/flan-t5-dialogue-summarizer/training/notebook.ipynb)

Fine-tunes [`google/flan-t5-base`](https://huggingface.co/google/flan-t5-base) with a **LoRA** adapter to summarize
everyday conversations from [DialogSum](https://huggingface.co/datasets/knkarthick/dialogsum). The adapter is compared,
on the same test dialogues, with **prompting** the frozen base model zero-shot, one-shot and few-shot. The same model
then answers questions about a fictional store from a small knowledge base (**RAG**: MiniLM retrieval, then grounded
generation). The result is served by the [Space](https://huggingface.co/spaces/shalev396/flan-t5-dialogue-summarizer) from the
[model repo](https://huggingface.co/shalev396/flan-t5-dialogue-summarizer).

All logic lives in `src/` and `../model/model.py`; this notebook calls it step by step.

1. Setup · 2. Config · 3. Data · 4. Load model · 5. Training · 6. Evaluation · 7. Inference · 8. Export

## 1. Setup
1a finds the project files (clones the repo on a fresh runtime), 1b installs the dependencies from the root `requirements.txt`, 1c picks the `device`.

In [ ]:
# 1a. Project + files — fetch the repo / model code only if they aren't here (e.g. a fresh Colab runtime)
import os, subprocess
from pathlib import Path

SLUG = "flan-t5-dialogue-summarizer"
if not Path("src").is_dir():                      # not inside <slug>/training yet -> get the repo
    if not Path("ml-lab").is_dir():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/shalev396/ml-lab.git"], check=True)
    os.chdir(f"ml-lab/{SLUG}/training")
if not Path("../model/model.py").exists():   # model/ is a git submodule (HF repo): code only, skip weights
    subprocess.run(["git", "submodule", "update", "--init", "--depth", "1", f"{SLUG}/model"],
                   cwd="../..", check=True, env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"})
print("working dir:", Path.cwd())

In [ ]:
# 1b. Dependencies — the root requirements.txt is the single source of truth for every project.
# Colab/Jupyter never install a repo's requirements on their own, so this cell does it (fast no-op when satisfied).
%pip install -q -r ../../requirements.txt
# Colab preinstalls torchao 0.10; peft refuses torchao < 0.16 when it adds LoRA. This project never uses torchao.
import importlib.metadata as _md
try:
    if tuple(int(x) for x in _md.version("torchao").split(".")[:2]) < (0, 16):
        %pip uninstall -y -q torchao
except _md.PackageNotFoundError:
    pass

In [ ]:
# 1c. Device — whatever this machine has (cuda -> mps -> cpu); nothing platform-specific is installed.
import torch
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", device)

In [ ]:
# 1d. Project code: src/ (training) + ../model/model.py (inference, shared with the Space)
import pandas as pd
from IPython.display import Image, Markdown, display

from src import data_setup, engine, export, model_builder, utils
from src.config import Config
import model as M

pd.set_option("display.max_colwidth", 200)
print(utils.lib_versions("torch", "transformers", "peft", "datasets", "sentence_transformers"))

## 2. Config
Every hyperparameter lives in `src/config.py`. `SMOKE_TEST=1` swaps in `flan-t5-small`, 64 training dialogues and
1 epoch, so the whole notebook runs in a few minutes on a CPU. Smoke runs export to `outputs/smoke/model/` and never
touch `../model/`.

The full run (flan-t5-base, all 12,460 train dialogues, up to 6 epochs of 1,558 steps) keeps the epoch with the lowest
validation loss and stops early after 2 epochs without a new best, so a late overfitting epoch is never exported.
Measured on an RTX 2080 Ti: 0.52 s/step (about 14 min per epoch); a Colab T4 is roughly 2x slower, an L4/A100 faster.
On a CPU it takes days. Epoch checkpoints in `outputs/checkpoints/` let an interrupted run continue where it stopped.

In [ ]:
import os
os.environ.setdefault("SMOKE_TEST", "0")   # "1" = quick sanity run (flan-t5-small, tiny subsets)
PUSH_TO_HUB = False                        # upload ../model/ to the Hub at the end (needs a write token)

cfg = Config()
utils.set_seeds(cfg.seed, "torch")
print("smoke:", cfg.smoke, "| base model:", cfg.base_model, "| export to:", cfg.model_dir.relative_to(utils.PROJECT_DIR))
print("training precision on this device:", engine.precision_name(device))
pd.Series(cfg.to_dict()).to_frame("value")

## 3. Data
DialogSum: 12,460 train, 500 validation and 500 test dialogues (the test dialogues have 3 human summaries each).
It is downloaded once to `data/hf_datasets/`, with the authors' GitHub JSONL files as a fallback. All of it is used:
the train split for fine-tuning, the validation split for the loss after each epoch (and the checkpoint choice), and
the test split for ROUGE, scored once at the end.

In [ ]:
ds = data_setup.load_dialogsum()
splits = data_setup.make_splits(ds, cfg)
example = splits.train[0]
print(example["dialogue"][:600], "\n\nHUMAN SUMMARY:", example["summary"])

In [ ]:
# Lengths in words: summaries are ~5x shorter than the dialogues they describe
lengths = pd.DataFrame({"dialogue words": [len(d.split()) for d in splits.train["dialogue"]],
                        "summary words": [len(s.split()) for s in splits.train["summary"]]})
lengths.describe().round(1).T

In [ ]:
# Tokenize (prompt -> input_ids, summary -> labels) and batch with per-batch dynamic padding
tokenizer = model_builder.load_tokenizer(cfg)
train_loader, val_loader = data_setup.make_loaders(splits, tokenizer, cfg)
print("prompt template:", repr(M.PROMPT_TEMPLATE))
print(f"{len(train_loader)} training batches of {cfg.batch_size} · {len(val_loader)} validation batches")

## 4. Load model
The pretrained FLAN-T5 (float32), then LoRA adapters of rank 16 on the attention query/value projections of every
layer. Only the adapters are trainable, about 0.7 % of the weights. At serving time `model.py` merges them into the
base weights, so inference costs exactly what the base model costs.

In [ ]:
base = model_builder.load_base(cfg, device)
peft_model = model_builder.add_lora(base, cfg)
params = model_builder.count_params(peft_model)
print(f"base parameters: {params['base']:,} · trainable LoRA parameters: {params['trainable']:,} "
      f"({100 * params['trainable'] / params['base']:.2f} %)")

## 5. Training
Cross-entropy on the summary tokens, AdamW (lr 1e-3, weight decay 0.01) with linear decay to 0, batch size 8,
gradient clipping at 1.0. Precision is bf16 autocast on GPUs that support it and float32 elsewhere. fp16 is never
used, because T5 overflows in fp16 and the loss becomes NaN.

**Why not RLHF?** Reinforcement learning from human feedback trains a reward model on human preference pairs,
then optimises the LLM against it (PPO, or DPO directly on the pairs). It is the step after supervised fine-tuning
that makes chat models helpful and harmless. DialogSum gives us reference summaries, not preference pairs, so we
stop at supervised fine-tuning.

In [ ]:
history = engine.train(peft_model, train_loader, val_loader, cfg, device)
print(f"trained in {history['train_time_s']:.0f} s ({history['seconds_per_step']:.2f} s per step, {history['precision']})")
export.plot_loss(history, cfg.outputs_dir / "loss_curve.png")
display(Image(filename=str(cfg.outputs_dir / "loss_curve.png"), width=640))

In [ ]:
# Save the servable model: adapter + tokenizer + config.json (what section 7 and the Space load)
export.save_model(cfg, peft_model, tokenizer);

## 6. Evaluation
Four variants on the same test dialogues, all with greedy decoding:
- **base zero-shot**: the frozen FLAN-T5 with the plain instruction.
- **base one-shot / few-shot**: the same model with 1 or 2 solved train examples in the prompt (in-context
  learning changes the prompt, not the weights).
- **LoRA fine-tuned**: the deployed model.

ROUGE-1/2/L F-measure. Each prediction is scored against the 3 human summaries of its dialogue, and the scores are
averaged. The base variants reuse the same network with the adapter switched off.

In [ ]:
shots = data_setup.pick_shots(splits.train, cfg)
results = engine.evaluate_variants(peft_model, tokenizer, splits, shots, cfg)
table = pd.DataFrame({name: {k: r[k] for k in ("rouge1", "rouge2", "rougeL", "gen_len_words", "seconds")}
                      for name, r in results.items()}).T
table[["rouge1", "rouge2", "rougeL"]] *= 100
table.round(2)

In [ ]:
# Side by side on a few test dialogues
pd.DataFrame({"dialogue": [d[:300] for d in splits.test["dialogue"][:3]],
              "human": [r[0] for r in splits.test["references"][:3]],
              **{name: r["predictions"][:3] for name, r in results.items()}}).T

### RAG: retrieval quality and grounded answers
The retriever embeds the 41 knowledge-base documents and the question with MiniLM and ranks the documents by cosine
similarity. It is scored on 18 held-out questions, each labelled with the document that answers it. The answers
compare the grounded prompt (top-3 documents as context) with the bare question.

In [ ]:
retriever = M.Retriever(data_setup.load_kb(), cfg.embed_model)
retrieval = engine.evaluate_retrieval(retriever, data_setup.RAG_EVAL, k=cfg.rag_top_k)
print(retrieval)
questions = ["How long does standard shipping take?", "Can I return earbuds after opening them?",
             "What does the NovaCare+ extended warranty cover?"]
pd.DataFrame(engine.rag_answers(peft_model, tokenizer, retriever, questions, cfg))

## 7. Inference
Exactly what the Space and the Inference Endpoint run: `model.load(dir, device)` downloads the base model, merges the
saved adapter on the CPU, and summarizes the example conversations from `../space/examples/`.

In [ ]:
predictor = M.load(cfg.model_dir, device)
for path in sorted((utils.SPACE_DIR / "examples").glob("*.txt")):
    print(f"{path.name}: {predictor.predict(path.read_text(encoding='utf-8'))}")
answer = predictor.rag("Can I pay in installments?")
print("\nRAG:", answer["answer"], "| sources:", [s["id"] for s in answer["sources"]])

## 8. Export
`metrics.json`, the card plots (`assets/`) and the model card are written next to the adapter. The round trip is
checked first: `model.py` reloads the export and must reproduce the in-memory summaries. `PUSH_TO_HUB = True` uploads
the folder to [shalev396/flan-t5-dialogue-summarizer](https://huggingface.co/shalev396/flan-t5-dialogue-summarizer);
the token comes from `hf auth login` or an `HF_TOKEN` secret / environment variable. Smoke runs are never uploaded.

In [ ]:
metrics = export.export(cfg, history=history, results=results, retrieval=retrieval, splits=splits,
                        params=params, device=str(device))
display(Markdown(export.results_markdown(metrics).split("![")[0]))
for name in ("rouge_comparison.png", "rouge_per_dialogue.png"):
    display(Image(filename=str(cfg.model_dir / "assets" / name), width=640))
sorted(p.relative_to(cfg.model_dir).as_posix() for p in cfg.model_dir.rglob("*") if p.is_file())

In [ ]:
if PUSH_TO_HUB and not cfg.smoke:
    print(utils.upload_model(cfg.model_dir))
else:
    print("not uploading (PUSH_TO_HUB is False or this is a smoke run)")